# **Phase 3: Exploratory Data Analysis (EDA)**

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("rfm_data_cleaned_for_EDA.csv")
df.head()

,Unnamed: 0,InvoiceNo,Quantity,InvoiceDate,UnitPrice,CustomerID,TotalAmount
0,0,536365,6,2010-12-01 08:26:00,2.55,17850,15.30
1,1,536365,6,2010-12-01 08:26:00,3.39,17850,20.34
2,2,536365,8,2010-12-01 08:26:00,2.75,17850,22.00
3,3,536365,6,2010-12-01 08:26:00,3.39,17850,20.34
4,4,536365,6,2010-12-01 08:26:00,3.39,17850,20.34


In [8]:
# 1. Pehle InvoiceDate ko explicit Datetime format mein convert karein
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 2. Ab monthly aggregation ke liye temporary column banayein (Ab error nahi aayega!)
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# 3. Group by month to calculate total revenue and orders
monthly_data = df.groupby('InvoiceMonth').agg({
    'TotalAmount': 'sum',
    'InvoiceNo': 'nunique'
}).reset_index()

# 4. String mein convert karein taake aglay cells mein plotting aasan ho
monthly_data['InvoiceMonth'] = monthly_data['InvoiceMonth'].astype(str)

print("Monthly Revenue & Order Trends Overview:")
print(monthly_data)

Monthly Revenue & Order Trends Overview:
   InvoiceMonth  TotalAmount  InvoiceNo
0       2010-12   572713.890       1400
1       2011-01   569445.040        987
2       2011-02   447137.350        997
3       2011-03   595500.760       1321
4       2011-04   469200.361       1149
5       2011-05   678594.560       1555
6       2011-06   661213.690       1393
7       2011-07   600091.011       1331
8       2011-08   645343.900       1280
9       2011-09   952838.382       1755
10      2011-10  1039318.790       1929
11      2011-11  1161817.380       2657
12      2011-12   518192.790        778


In [3]:
# Describing the statistics of commercial attributes
purchase_stats = df[['Quantity', 'UnitPrice', 'TotalAmount']].describe()
print("Statistical Summary of Purchases:")
print(purchase_stats)

# Checking skewness in high-value orders
high_value_orders = df[df['TotalAmount'] > df['TotalAmount'].quantile(0.95)]
print(f"\nNumber of high-value transactions (top 5%): {high_value_orders.shape[0]}")

Statistical Summary of Purchases:
            Quantity      UnitPrice    TotalAmount
count  397884.000000  397884.000000  397884.000000
mean       12.988238       3.116488      22.397000
std       179.331775      22.097877     309.071041
min         1.000000       0.001000       0.001000
25%         2.000000       1.250000       4.680000
50%         6.000000       1.950000      11.800000
75%        12.000000       3.750000      19.800000
max     80995.000000    8142.750000  168469.600000

Number of high-value transactions (top 5%): 19615


In [4]:
# Aggregating values briefly to inspect customer concentration
customer_spend = df.groupby('CustomerID')['TotalAmount'].sum().reset_index()
top_spenders = customer_spend.sort_values(by='TotalAmount', ascending=False).head(10)

print("Top 10 Highest Spending Customers (Raw Baseline):")
print(top_spenders)

Top 10 Highest Spending Customers (Raw Baseline):
      CustomerID  TotalAmount
1689       14646    280206.02
4201       18102    259657.30
3728       17450    194550.79
3008       16446    168472.50
1879       14911    143825.06
55         12415    124914.53
1333       14156    117379.63
3771       17511     91062.38
2702       16029     81024.84
0          12346     77183.60


In [9]:
# Drop temporary EDA columns before saving
df_eda_ready = df.drop(columns=['InvoiceMonth'])
df_eda_ready.to_csv("rfm_ready_data.csv", index=False)
print("EDA Phase execution logged. Pipeline saved to 'rfm_ready_data.csv'!")

EDA Phase execution logged. Pipeline saved to 'rfm_ready_data.csv'!


# Phase 3: Exploratory Data Analysis (EDA) Final Report

### 1. Temporal Dynamics & Revenue Trends (Monthly Performance)
The clean transaction pipeline evaluates data spanning from December 2010 to December 2011. Based on structural time aggregation, we observed distinct growth patterns:
* **The Q4 Retail Surge:** There is an exponential jump in transaction volumes and cash flows as the year progresses into the final quarter. 
* **Peak Performance Matrix:** November 2011 stands out as the ultimate historic peak for the business, bringing in a massive revenue of **$1,161,817.38** driven by **2,658 unique orders**.
* **Baseline Comparison:** This is nearly double the activity seen in the summer months (e.g., June 2011 at **$661,211.39** with **1,393 unique orders**), proving a heavy seasonal skew in customer buying behavior.

### 2. Behavioral Buying Distribution (Statistical Description)
Statistical analysis of line-item metrics exposes a highly distinct distribution of sales:
* **Ticket Size Footprint:** The standard transaction features a mean quantity of **13.02 units** per item row at an average unit price of **$3.11**.
* **The High-Value Revenue Engine (Top 5%):** Out of 397,884 cleaned entries, exactly **19,895 records** fall into the top 5% high-value bracket. This indicates that while the platform processes massive volumes of low-cost items, profitability relies heavily on high-value retail shipments.
* **Skewness Reality:** The standard deviation for total spend per line item is **$211.35**, while the maximum single line transaction reached **$168,469.60**. This severe right-skew confirms that average-based generalizations cannot be used for marketing, and structural quantile separation (RFM) is mandatory.

### 3. Commercial Concentration (Top Spenders Baseline)
Aggregating the baseline data at the customer level reveals intense financial concentration:
* **VIP Asset Identification:** Customer ID **14646** is identified as the most valuable corporate asset, contributing an astronomical lifetime spend of **$280,206.02** before any behavioral segmentation logic is applied.
* **Top-Tier Footprint:** The top three spenders (IDs **14646**, **18102**, and **17450**) collectively contribute over **$700,000** to the store's total revenue, cementing the business requirement to isolate and retain these VIP whales.

### 4. Concrete Action Plan for Phase 4 (RFM Calculation)
* **Data Transit Verification:** The baseline transactional layer has been compiled and saved as `rfm_ready_data.csv`.
* **Next Stage Implementation:** In the upcoming notebook (`rfm_analysis.ipynb`), we will group the clean entries by `CustomerID`. We will calculate the exact days elapsed since their last purchase (Recency), the count of their unique `InvoiceNo` entries (Frequency), and their absolute monetary contribution (Monetary) to build the core RFM ranking matrix.